Phase 1

In [ ]:
!pip -q install asteroid
!pip -q install mir_eval
!pip -q install pesq
!pip -q install pystoi
!pip -q install torchmetrics

In [ ]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchaudio

from asteroid.models import ConvTasNet
from asteroid.losses import PITLossWrapper
from asteroid.losses.sdr import PairwiseNegSDR

import mir_eval
from pesq import pesq
from pystoi import stoi

warnings.filterwarnings("ignore")

torch.backends.cudnn.benchmark = True

In [ ]:
# ======================================================
# Dataset Paths
# ======================================================

TRAIN_PATH = "/kaggle/input/datasets/unconscious/libri2mix-8khz-min/Libri2Mix/wav8k/min/train-100"

VALID_PATH = "/kaggle/input/datasets/unconscious/libri2mix-8khz-min/Libri2Mix/wav8k/min/dev"

TEST_PATH = "/kaggle/input/datasets/unconscious/libri2mix-8khz-min/Libri2Mix/wav8k/min/test"

# ======================================================
# Audio
# ======================================================

SAMPLE_RATE = 8000

SEGMENT_SECONDS = 4

SEGMENT_LENGTH = SAMPLE_RATE * SEGMENT_SECONDS

# ======================================================
# Training
# ======================================================

BATCH_SIZE = 8

NUM_WORKERS = 2

EPOCHS = 40

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT_PATH = "best_convtasnet.pt"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device :", DEVICE)

In [ ]:
class Libri2MixDataset(Dataset):

    def __init__(
        self,
        root_dir,
        segment_length=None,
        training=False,
    ):

        self.root_dir = Path(root_dir)

        self.training = training

        self.segment_length = segment_length

        self.mix_dir = self.root_dir / "mix_clean"

        self.s1_dir = self.root_dir / "s1"

        self.s2_dir = self.root_dir / "s2"

        assert self.mix_dir.exists(), f"{self.mix_dir} not found"

        assert self.s1_dir.exists(), f"{self.s1_dir} not found"

        assert self.s2_dir.exists(), f"{self.s2_dir} not found"

        self.files = sorted(os.listdir(self.mix_dir))

        print(f"{root_dir}")

        print(f"Loaded {len(self.files)} mixtures")

    def __len__(self):

        return len(self.files)

    def random_crop(self, mix, s1, s2):

        length = mix.shape[-1]

        if length <= self.segment_length:

            pad = self.segment_length - length

            if pad > 0:

                mix = torch.nn.functional.pad(mix, (0, pad))

                s1 = torch.nn.functional.pad(s1, (0, pad))

                s2 = torch.nn.functional.pad(s2, (0, pad))

            return mix, s1, s2

        start = random.randint(

            0,

            length - self.segment_length

        )

        end = start + self.segment_length

        return (

            mix[:, start:end],

            s1[:, start:end],

            s2[:, start:end],

        )

    def __getitem__(self, idx):

        filename = self.files[idx]

        mix, sr = torchaudio.load(

            self.mix_dir / filename

        )

        s1, _ = torchaudio.load(

            self.s1_dir / filename

        )

        s2, _ = torchaudio.load(

            self.s2_dir / filename

        )

        if self.training:

            mix, s1, s2 = self.random_crop(

                mix,

                s1,

                s2,

            )

        target = torch.stack(

            [

                s1.squeeze(0),

                s2.squeeze(0),

            ],

            dim=0,

        )

        return (

            mix.squeeze(0),

            target,

        )

In [ ]:
train_dataset = Libri2MixDataset(

    TRAIN_PATH,

    segment_length=SEGMENT_LENGTH,

    training=True,

)

valid_dataset = Libri2MixDataset(

    VALID_PATH,

    training=False,

)

test_dataset = Libri2MixDataset(

    TEST_PATH,

    training=False,

)

In [ ]:
train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=True,

)

valid_loader = DataLoader(

    valid_dataset,

    batch_size=1,

    shuffle=False,

    num_workers=NUM_WORKERS,

)

test_loader = DataLoader(

    test_dataset,

    batch_size=1,

    shuffle=False,

    num_workers=NUM_WORKERS,

) 

In [ ]:
print("=" * 60)

print("Training Samples :", len(train_dataset))

print("Validation Samples :", len(valid_dataset))

print("Testing Samples :", len(test_dataset))

print("=" * 60)

mix, target = train_dataset[0]

print("Mixture Shape :", mix.shape)

print("Target Shape :", target.shape)

print("Sample Rate :", SAMPLE_RATE)

Phase 2

In [ ]:
# ======================================================
# Conv-TasNet
# ======================================================

model = ConvTasNet(
    n_src=2,
    sample_rate=SAMPLE_RATE,
)

model = model.to(DEVICE)

print(model)

In [ ]:
# ======================================================
# PIT SI-SDR Loss
# ======================================================

criterion = PITLossWrapper(
    PairwiseNegSDR("sisdr"),
    pit_from="pw_mtx",
)

In [ ]:
# ======================================================
# Optimizer
# ======================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
)

In [ ]:
# ======================================================
# Mixed Precision
# ======================================================

scaler = torch.cuda.amp.GradScaler()

best_val_loss = float("inf")

history = {
    "train_loss": [],
    "val_loss": [],
    "sisdri": [],
}

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")

In [ ]:
mix, target = next(iter(train_loader))

mix = mix.to(DEVICE)

target = target.to(DEVICE)

print("Input Mix :", mix.shape)
print("Target    :", target.shape)

with torch.no_grad():

    output = model(mix)

print("Prediction:", output.shape)

loss = criterion(
    output,
    target,
)

print("Loss:", loss.item())

In [ ]:
if DEVICE == "cuda":

    print(
        f"Allocated : {torch.cuda.memory_allocated()/1024**2:.2f} MB"
    )

    print(
        f"Reserved  : {torch.cuda.memory_reserved()/1024**2:.2f} MB"
    )

Phase 3

In [ ]:
def si_sdr(reference, estimation):

    reference = reference - reference.mean()
    estimation = estimation - estimation.mean()

    projection = (
        torch.sum(reference * estimation)
        * reference
        / (torch.sum(reference ** 2) + 1e-8)
    )

    noise = estimation - projection

    ratio = (
        torch.sum(projection ** 2)
        /
        (torch.sum(noise ** 2) + 1e-8)
    )

    return 10 * torch.log10(ratio + 1e-8)

In [ ]:
import itertools

def compute_sisdri(mixture, prediction, target):
    """
    PIT-aware SI-SDR Improvement.
    """

    mixture = mixture.cpu()
    prediction = prediction.cpu()
    target = target.cpu()

    best_score = -1e9

    for perm in itertools.permutations(range(2)):

        estimate_scores = []
        mixture_scores = []

        for pred_idx, tgt_idx in enumerate(perm):

            estimate_scores.append(
                si_sdr(
                    target[tgt_idx],
                    prediction[pred_idx],
                ).item()
            )

            mixture_scores.append(
                si_sdr(
                    target[tgt_idx],
                    mixture,
                ).item()
            )

        score = np.mean(estimate_scores) - np.mean(mixture_scores)

        if score > best_score:
            best_score = score

    return best_score

In [ ]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0

    progress = tqdm(loader)

    for mixture, target in progress:

        mixture = mixture.to(
            DEVICE,
            non_blocking=True,
        )

        target = target.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():

            prediction = model(mixture)

            loss = criterion(
                prediction,
                target,
            )

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.3f}"
        )

    return running_loss / len(loader)

In [ ]:
@torch.no_grad()

def validate(model, loader):

    model.eval()

    running_loss = 0

    sisdri_scores = []

    progress = tqdm(loader)

    for mixture, target in progress:

        mixture = mixture.to(DEVICE)

        target = target.to(DEVICE)

        prediction = model(mixture)

        loss = criterion(
            prediction,
            target,
        )

        running_loss += loss.item()

        sisdri = compute_sisdri(

            mixture[0],

            prediction[0],

            target[0],

        )

        sisdri_scores.append(sisdri)

    return (

        running_loss / len(loader),

        np.mean(sisdri_scores),

    )

In [ ]:
for epoch in range(EPOCHS):

    print("=" * 70)

    print(f"Epoch {epoch+1}/{EPOCHS}")

    train_loss = train_one_epoch(
        model,
        train_loader,
    )

    val_loss, sisdri = validate(
        model,
        valid_loader,
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)

    history["val_loss"].append(val_loss)

    history["sisdri"].append(sisdri)

    print()

    print(f"Train Loss : {train_loss:.4f}")

    print(f"Val Loss   : {val_loss:.4f}")

    print(f"SI-SDRi    : {sisdri:.2f} dB")

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "loss": val_loss,
            },
            CHECKPOINT_PATH,
        )

        print()

        print("✓ Saved Best Model")

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
)

model.load_state_dict(
    checkpoint["model"]
)

optimizer.load_state_dict(
    checkpoint["optimizer"]
)

print("Loaded checkpoint")

print()

print("Epoch :", checkpoint["epoch"] + 1)

print("Validation Loss :", checkpoint["loss"])

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history["train_loss"],
    label="Train",
)

plt.plot(
    history["val_loss"],
    label="Validation",
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training Curve")

plt.legend()

plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history["sisdri"])

plt.xlabel("Epoch")

plt.ylabel("SI-SDR Improvement (dB)")

plt.title("Validation SI-SDRi")

plt.grid()

plt.show()

Phase 4

In [ ]:
def compute_si_sdr(reference, estimate):

    scores = []

    for i in range(2):

        score = si_sdr(
            reference[i],
            estimate[i],
        )

        scores.append(score.item())

    return np.mean(scores)

In [ ]:
def compute_sdr(reference, estimate):

    reference = reference.numpy()

    estimate = estimate.numpy()

    sdr, _, _, _ = mir_eval.separation.bss_eval_sources(
        reference,
        estimate,
    )

    return np.mean(sdr)

In [ ]:
def compute_pesq(reference, estimate):

    scores = []

    for i in range(2):

        try:

            score = pesq(
                SAMPLE_RATE,
                reference[i].numpy(),
                estimate[i].numpy(),
                "nb",
            )

            scores.append(score)

        except:

            pass

    if len(scores) == 0:

        return 0

    return np.mean(scores)

In [ ]:
def compute_stoi(reference, estimate):

    scores = []

    for i in range(2):

        score = stoi(
            reference[i].numpy(),
            estimate[i].numpy(),
            SAMPLE_RATE,
            extended=False,
        )

        scores.append(score)

    return np.mean(scores)

In [ ]:
@torch.no_grad()

def evaluate(model, loader):

    model.eval()

    sisdr_scores = []

    sisdri_scores = []

    sdr_scores = []

    pesq_scores = []

    stoi_scores = []

    progress = tqdm(loader)

    for mixture, target in progress:

        mixture = mixture.to(DEVICE)

        target = target.to(DEVICE)

        prediction = model(mixture)

        pred = prediction.squeeze(0).cpu()

        tgt = target.squeeze(0).cpu()

        mix = mixture.squeeze(0).cpu()

        sisdr_scores.append(
            compute_si_sdr(
                tgt,
                pred,
            )
        )

        sisdri_scores.append(
            compute_sisdri(
                mix,
                pred,
                tgt,
            )
        )

        sdr_scores.append(
            compute_sdr(
                tgt,
                pred,
            )
        )

        pesq_scores.append(
            compute_pesq(
                tgt,
                pred,
            )
        )

        stoi_scores.append(
            compute_stoi(
                tgt,
                pred,
            )
        )

    results = {

        "SI-SDR": np.mean(sisdr_scores),

        "SI-SDRi": np.mean(sisdri_scores),

        "SDR": np.mean(sdr_scores),

        "PESQ": np.mean(pesq_scores),

        "STOI": np.mean(stoi_scores),

    }

    return results

In [ ]:
results = evaluate(

    model,

    test_loader,

)

metrics = pd.DataFrame(

    results,

    index=["ConvTasNet"],

)

metrics

In [ ]:
metrics.to_csv(

    "metrics.csv",

    index=False,

)

metrics

Phase 5

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
)

model.load_state_dict(
    checkpoint["model"]
model.load_state_dict(
    checkpoint["model"]
)
    
model.eval()

print("Best model loaded.")

In [ ]:
sample_audio = os.path.join(
    TEST_PATH,
    "mix_clean",
    os.listdir(
        os.path.join(TEST_PATH, "mix_clean")
    )[0],
)

print(sample_audio)

In [ ]:
waveform, sr = torchaudio.load(sample_audio)

assert sr == SAMPLE_RATE

waveform = waveform.to(DEVICE)

with torch.no_grad():

    separated = model(
        waveform
    )

separated = separated.squeeze(0).cpu()

print(separated.shape)

In [ ]:
os.makedirs(
    "outputs",
    exist_ok=True,
)

torchaudio.save(

    "outputs/speaker1.wav",

    separated[0].unsqueeze(0),

    SAMPLE_RATE,

)

torchaudio.save(

    "outputs/speaker2.wav",

    separated[1].unsqueeze(0),

    SAMPLE_RATE,

)

print("Done.")

In [ ]:
from IPython.display import Audio, display

print("Original Mixture")

display(Audio(sample_audio))

print("Speaker 1")

display(Audio("outputs/speaker1.wav"))

print("Speaker 2")

display(Audio("outputs/speaker2.wav"))

In [ ]:
fig, ax = plt.subplots(
    3,
    1,
    figsize=(16,8),
)

ax[0].plot(
    waveform.cpu().squeeze().numpy()
)

ax[0].set_title("Mixture")

ax[1].plot(
    separated[0].numpy()
)

ax[1].set_title("Speaker 1")

ax[2].plot(
    separated[1].numpy()
)

ax[2].set_title("Speaker 2")

plt.tight_layout()

plt.show()

In [ ]:
spec = torchaudio.transforms.Spectrogram()

fig, ax = plt.subplots(
    3,
    1,
    figsize=(16,10),
)

ax[0].imshow(
    torch.log1p(
        spec(
            waveform.cpu()
        )
    )[0],
    origin="lower",
    aspect="auto",
)

ax[0].set_title("Mixture")

ax[1].imshow(
    torch.log1p(
        spec(
            separated[0].unsqueeze(0)
        )
    )[0],
    origin="lower",
    aspect="auto",
)

ax[1].set_title("Speaker 1")

ax[2].imshow(
    torch.log1p(
        spec(
            separated[1].unsqueeze(0)
        )
    )[0],
    origin="lower",
    aspect="auto",
)

ax[2].set_title("Speaker 2")

plt.tight_layout()

plt.show()

In [ ]:
output_dir = "batch_outputs"

os.makedirs(
    output_dir,
    exist_ok=True,
)

mix_dir = os.path.join(
    TEST_PATH,
    "mix_clean",
)

files = sorted(
    os.listdir(mix_dir)
)

model.eval()

with torch.no_grad():

    for file in tqdm(files):

        wav, sr = torchaudio.load(
            os.path.join(
                mix_dir,
                file,
            )
        )

        wav = wav.to(DEVICE)

        pred = model(wav)

        pred = pred.squeeze(0).cpu()

        torchaudio.save(

            os.path.join(
                output_dir,
                file.replace(
                    ".wav",
                    "_s1.wav",
                ),
            ),

            pred[0].unsqueeze(0),

            SAMPLE_RATE,

        )

        torchaudio.save(

            os.path.join(
                output_dir,
                file.replace(
                    ".wav",
                    "_s2.wav",
                ),
            ),

            pred[1].unsqueeze(0),

            SAMPLE_RATE,

        )

print("Finished.")